# Connect 4 Reinforcement Learning Tutorial

This notebook is the main teaching artifact for the assignment tutorial. It introduces Connect 4 as a reinforcement learning problem, shows the rule-correct simulator and Gymnasium environment, records the baseline training configuration, and reserves space for later evaluation plots and discussion.

## RL Formulation

Connect 4 maps cleanly to reinforcement learning:

- Agent: the player-one policy being trained
- Environment: the Connect 4 Gymnasium wrapper built on top of the simulator
- State: board state, active player, terminal flag, winner, and last move
- Observation: the 6x7 board plus the current-player indicator
- Action: choosing one of the seven columns
- Reward: sparse feedback for win, loss, draw, invalid move, or neutral progress
- Policy: the rule used by the agent to select a column from an observation

The baseline setup keeps the agent as player one and uses a random legal-move opponent.

In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np

from connect4 import Connect4Game
from connect4_env import Connect4Env

PROJECT_ROOT = Path.cwd()
SEED = 7

random.seed(SEED)
np.random.seed(SEED)

print(f'Project root: {PROJECT_ROOT}')
print(f'Seed: {SEED}')

## Simulator Walkthrough

The simulator is the source of truth for game rules. It enforces board size, gravity, alternating turns, invalid moves, wins, draws, and terminal-state blocking.

In [ ]:
game = Connect4Game()
for move in [3, 2, 3, 2, 3]:
    game.apply_move(move)

print(game.render())
print()
print('Current player:', game.get_current_player())
print('Legal actions:', game.get_legal_actions())
print('Terminal:', game.is_terminal())

## Gymnasium Environment Walkthrough

The environment wraps the simulator so training code can call `reset()` and `step()` using Gymnasium conventions. Each step represents one agent move and, if the game continues, one opponent response.

In [ ]:
env = Connect4Env()
observation, info = env.reset(seed=SEED)

print('Initial observation keys:', list(observation.keys()))
print('Current player:', observation['current_player'])
print('Legal actions:', info['legal_actions'])

next_observation, reward, terminated, truncated, next_info = env.step(3)
print('Reward:', reward)
print('Terminated:', terminated)
print('Truncated:', truncated)
print('Opponent action:', next_info['opponent_action'])
print(env.render())

## Training Configuration

Phase 8 records the baseline training setup without running a full training session yet. The first end-to-end training run will be carried out in Phase 9.

In [ ]:
TRAINING_LIBRARIES = {
    'environment': ['gymnasium', 'numpy'],
    'visualization': ['matplotlib'],
    'planned_training_stack': ['stable-baselines3', 'torch'],
}

BASELINE_TRAINING_CONFIG = {
    'seed': SEED,
    'agent_role': 'player_one',
    'opponent': 'random_legal_action',
    'observation_format': 'dict(board, current_player)',
    'action_space': 'Discrete(7)',
    'reward_scheme': {
        'win': 1.0,
        'loss': -1.0,
        'draw': 0.0,
        'valid_non_terminal': 0.0,
        'illegal_move': -1.0,
    },
    'action_masking': 'deferred',
    'phase_9_goal': 'choose and run a baseline RL algorithm',
}

TRAINING_LIBRARIES, BASELINE_TRAINING_CONFIG

## Evaluation and Plot Plan

The notebook will later plot baseline learning evidence such as episode reward, win rate, draw rate, and invalid-action count. This section prepares the plotting structure before the first training run is added.

In [ ]:
episode_rewards = []
win_rates = []
draw_rates = []
invalid_action_counts = []

def plot_training_metrics():
    figure, axes = plt.subplots(2, 2, figsize=(10, 6))
    metric_series = [
        ('Episode Reward', episode_rewards),
        ('Win Rate', win_rates),
        ('Draw Rate', draw_rates),
        ('Invalid Action Count', invalid_action_counts),
    ]

    for axis, (title, values) in zip(axes.flat, metric_series):
        axis.plot(values)
        axis.set_title(title)
        axis.set_xlabel('Evaluation Window')

    figure.tight_layout()
    return figure

print('Metric buffers initialized for Phase 9 training outputs.')

## Limitations and Future Directions

Current limitations:

- no training run has been executed yet
- the notebook does not yet include learned policy evaluation
- action masking is still deferred
- the first opponent is intentionally weak

Planned next steps:

- choose the first RL algorithm in Phase 9
- run a baseline training experiment against the random opponent
- populate the metric buffers and plots
- evaluate the trained policy with sample games and summary statistics